# ST-GCN 베이스라인 학습 (KSL 67-class)

이 노트북은 VSCode(로컬, CPU/MPS)와 Colab(T4 GPU) **양쪽에서 그대로 실행**되도록 작성되었습니다.

구성:
1. 환경 감지 (Colab vs 로컬)
2. 의존성 설치 + 경로 설정
3. 데이터셋 정합성 점검
4. 학습 실행 (`train_stgcn.py` 호출)
5. 학습 곡선 시각화
6. 예측 데모

데이터/스크립트는 README 정합성 검증 결과(train 973 / val 133 / test 117, 67 classes)를 그대로 사용합니다.

## 1. 환경 감지

In [ ]:
import os, sys, platform

IN_COLAB = "google.colab" in sys.modules
print(f"Colab: {IN_COLAB}")
print(f"Python: {platform.python_version()}")

try:
    import torch
    print(f"torch: {torch.__version__}")
    if torch.cuda.is_available():
        print(f"CUDA: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
    elif torch.backends.mps.is_available():
        print("MPS available (Apple Silicon)")
    else:
        print("GPU 없음 — CPU 사용")
except ImportError:
    print("torch 미설치 — 다음 셀에서 설치")

## 2. 의존성 설치 & 프로젝트 경로

**Colab**: Google Drive에 PSYcho 폴더가 있다고 가정. 경로가 다르면 `PROJECT_DIR` 만 수정.

**로컬(VSCode)**: 이 노트북이 `ai_engine/training/` 안에 있으면 자동으로 프로젝트 루트를 찾음.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/PSYcho"   # ← 본인 Drive 경로로 수정
else:
    PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

print(f"PROJECT_DIR = {PROJECT_DIR}")
assert os.path.isdir(PROJECT_DIR), f"경로 확인 필요: {PROJECT_DIR}"
os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, "ai_engine"))
print("cwd:", os.getcwd())

In [ ]:
%pip install -q torch numpy matplotlib

## 3. 데이터셋 정합성 빠른 점검

필요한 파일이 모두 제 위치에 있는지 확인.

In [ ]:
REQUIRED = [
    "ai_engine/data/keypoints/label_map.txt",
    "ai_engine/data/keypoints/000",
    "split_result.csv",
    "class_label.p",
    "ai_engine/training/dataset.py",
    "ai_engine/training/train_stgcn.py",
    "ai_engine/models/stgcn.py",
]
missing = [p for p in REQUIRED if not os.path.exists(p)]
if missing:
    print("❌ 누락된 파일/폴더:")
    for p in missing:
        print("  -", p)
    raise SystemExit("필요 파일을 먼저 업로드/배치하세요.")
print("✅ 모든 필수 파일 존재")

In [ ]:
from training.dataset import KSLKeypointDataset, load_class_words

DS_KW = dict(
    split_csv="split_result.csv",
    keypoints_dir="ai_engine/data/keypoints",
    label_map_path="ai_engine/data/keypoints/label_map.txt",
    t_fixed=105,
)
for sp in ("train", "val", "test"):
    d = KSLKeypointDataset(split=sp, **DS_KW)
    print(f"{sp:>5}: n={len(d):>4d}, dropped(zero-hand)={d._dropped_zero_hand}")

x, y = d[0]
print(f"sample shape: {tuple(x.shape)}, label_idx={y}")

vocab = load_class_words("class_label.p", d.num_classes)
print(f"vocab[0..4]: {vocab[:5]}")

## 4. 학습 실행

GPU 있으면 자동으로 cuda 사용, 없으면 CPU/MPS. Colab T4 기준 80 epoch ≈ **약 7분** 예상.

In [ ]:
EPOCHS = 80
BATCH = 64 if torch.cuda.is_available() else 32
WORKERS = 2 if IN_COLAB else 0
OUT_DIR = "weights/stgcn_baseline"

cmd = (
    f"python ai_engine/training/train_stgcn.py "
    f"--keypoints_dir ai_engine/data/keypoints "
    f"--split_csv split_result.csv "
    f"--label_map ai_engine/data/keypoints/label_map.txt "
    f"--class_pickle class_label.p "
    f"--out_dir {OUT_DIR} "
    f"--epochs {EPOCHS} --batch_size {BATCH} --num_workers {WORKERS} --device auto"
)
print(cmd)
!{cmd}

## 5. 학습 곡선 시각화

In [ ]:
import json
import matplotlib.pyplot as plt

with open(f"{OUT_DIR}/history.json") as f:
    hist = json.load(f)
with open(f"{OUT_DIR}/test_result.json") as f:
    test_res = json.load(f)

epochs = [h["epoch"] for h in hist]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(epochs, [h["train_loss"] for h in hist], label="train")
ax[0].plot(epochs, [h["val_loss"]   for h in hist], label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(epochs, [h["train_acc"] for h in hist], label="train")
ax[1].plot(epochs, [h["val_acc"]   for h in hist], label="val")
ax[1].axhline(test_res["test_acc"], color="red", linestyle="--",
              label=f"test={test_res['test_acc']:.3f}")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f"best val_acc = {test_res['best_val_acc']:.3f}")
print(f"test acc     = {test_res['test_acc']:.3f}")

## 6. 예측 데모

test 세트에서 몇 개 뽑아 모델 예측을 확인.

In [ ]:
import torch, random
from models.stgcn import STGCN

device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
test_ds = KSLKeypointDataset(split="test", **DS_KW)

model = STGCN(num_classes=test_ds.num_classes, mode="classify").to(device)
model.load_state_dict(torch.load(f"{OUT_DIR}/stgcn_best.pt", map_location=device))
model.eval()

random.seed(0)
sample_ids = random.sample(range(len(test_ds)), k=min(10, len(test_ds)))

with torch.no_grad():
    print(f"{'true':<20} | {'pred':<20} | top-1 prob")
    print("-" * 60)
    correct = 0
    for i in sample_ids:
        x, y = test_ds[i]
        logits = model(x.unsqueeze(0).to(device))
        prob = torch.softmax(logits, dim=-1)
        pred = prob.argmax(-1).item()
        ok = "O" if pred == y else "X"
        correct += int(pred == y)
        print(f"{vocab[y]:<20} | {vocab[pred]:<20} | {prob[0, pred].item():.3f}  {ok}")
    print(f"\n{correct}/{len(sample_ids)} 정답")